# Colab Quickstart

In [1]:
!git clone --recursive https://github.com/sapeirone/aml-2025-mistake-detection.git code

Cloning into 'code'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 437 (delta 5), reused 5 (delta 5), pack-reused 428 (from 1)
Receiving objects: 100% (437/437), 106.53 KiB | 5.92 MiB/s, done.
Resolving deltas: 100% (286/286), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 10.30 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# TODO: update the paths below with your own

!mkdir -p code/data
!unzip -q -o "/content/drive/MyDrive/AML_Project/1s.zip" -d code/data/

# Handle the nested zip structure: 1s/video/omnivore.zip
import os
import shutil
import glob

nested_zip = "code/data/1s/video/omnivore.zip"
target_dir = "code/data/video/omnivore"

if os.path.exists(nested_zip):
    !mkdir -p {target_dir}
    !unzip -q -o "{nested_zip}" -d code/data/temp_extract

    for f in glob.glob("code/data/temp_extract/**/*.npz", recursive=True):
        shutil.move(f, target_dir)

    !rm -rf code/data/1s code/data/temp_extract
    print("✅ Features setup complete.")

!mkdir -p code/checkpoints
!unzip -q -o "/content/drive/MyDrive/AML_Project/error_recognition_best.zip" -d code/checkpoints/

✅ Features setup complete.


In [4]:
!pip install torcheval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 8.8 MB/s eta 0:00:00


In [5]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ You are using a CPU runtime. Please go to 'Runtime > Change runtime type' and select 'T4 GPU' to fix the 'Torch not compiled with CUDA enabled' error.")
else:
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")

✅ GPU Detected: Tesla T4


In [6]:
%%bash

cd code
python -m core.evaluate --variant MLP --backbone omnivore \
  --ckpt checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_step_epoch_43.pt \
  --split step --threshold 0.6

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.4096162736939436, 'recall': 0.2989708115404083, 'f1': 0.3456549302643129, 'accuracy': 0.6831416629277163, 'auc': np.float64(0.6541560352028618), 'pr_auc': tensor(0.3187)}
test Step Level Metrics: {'precision': 0.6607142857142857, 'recall': 0.14859437751004015, 'f1': 0.24262295081967214, 'accuracy': 0.7105263157894737, 'auc': np.float64(0.7573902166041213), 'pr_auc': tensor(0.3638)}
----------------------------------------------------------------


test Progress: 42347/798: 100%|██████████| 798/798 [00:05<00:00, 149.48it/s]


In [10]:
import glob
import os
import subprocess

# For the 'recordings' split, threshold of 0.5 works better than the 0.4 that stated in the README.
configs = [
    # 1. MLP - Step Split (The one you just ran)
    {"variant": "MLP", "split": "step", "threshold": 0.6, "pattern": "*MLP*step*.pt"},

    # 2. MLP - Recordings Split
    {"variant": "MLP", "split": "recordings", "threshold": 0.5, "pattern": "*MLP*recordings*.pt"},

    # 3. Transformer - Step Split
    {"variant": "Transformer", "split": "step", "threshold": 0.6, "pattern": "*Transformer*step*.pt"},

    # 4. Transformer - Recordings Split
    {"variant": "Transformer", "split": "recordings", "threshold": 0.5, "pattern": "*Transformer*recordings*.pt"},
]

base_ckpt_dir = "code/checkpoints/error_recognition_best"

print("Starting Full Baseline Reproduction...")

for conf in configs:
    variant = conf["variant"]
    split = conf["split"]
    threshold = conf["threshold"]
    pattern = conf["pattern"]

    # Find the checkpoint file dynamically
    # We search in both 'MLP' and 'Transformer' directories
    search_path = os.path.join(base_ckpt_dir, variant, "omnivore", pattern)
    files = glob.glob(search_path)

    if not files:
        print(f"SKIPPING: Could not find checkpoint for {variant} on {split} split.")
        print(f"Searched for: {search_path}")
        continue

    ckpt_path = files[0] # Take the first match
    # The script runs inside 'code/', so we need the path relative to 'code/'
    relative_ckpt_path = ckpt_path.replace("code/", "")

    print(f"\n{'='*60}")
    print(f"▶️  Running: {variant} | Split: {split} | Threshold: {threshold}")
    print(f"    Checkpoint: {relative_ckpt_path}")
    print(f"{'='*60}\n")

    # Construct command
    cmd = [
        "python", "-m", "core.evaluate",
        "--variant", variant,
        "--backbone", "omnivore",
        "--ckpt", relative_ckpt_path,
        "--split", split,
        "--threshold", str(threshold)
    ]

    # Run command inside the 'code' directory
    try:
        process = subprocess.run(cmd, cwd="code", capture_output=True, text=True)

        # Print the output
        print(process.stdout)

        if process.returncode != 0:
            print("❌ Error occurred:")
            print(process.stderr)
    except Exception as e:
        print(f"❌ Execution failed: {e}")

print("\n✅ Reproduction sequence complete.")

Starting Full Baseline Reproduction...

▶️  Running: MLP | Split: step | Threshold: 0.6
    Checkpoint: checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_step_epoch_43.pt

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.4096162736939436, 'recall': 0.2989708115404083, 'f1': 0.3456549302643129, 'accuracy': 0.6831416629277163, 'auc': np.float64(0.6541560352028618), 'pr_auc': tensor(0.3187)}
test Step Level Metrics: {'precision': 0.6607142857142857, 'recall': 0.14859437751004015, 'f1': 0.24262295081967214, 'accuracy': 0.7105263157894737, 'auc': np.float64(0.7573902166041213), 'pr_auc': tensor(0.3638)}
----------------------------------------------------------------


▶️  Running: MLP | Split: recordings | Threshold: 0.5
    Checkpoint: checkpoints/error_recognition_best/MLP/omnivore/error_recognition_MLP_omnivore_rec